# M3 - Transfer Learning với MobileNetV2


Pipeline gồm:
1. Nạp trực tiếp ảnh RGB 224x224 đã lưu từ bước preprocessing.
2. Huấn luyện classifier với backbone MobileNetV2 được đóng băng.
3. Fine-tune các block cuối của backbone.
4. Chọn checkpoint tốt nhất trên validation và đánh giá một lần trên test.

Có thể điều chỉnh các siêu tham số trong cell **Cấu hình** trước khi huấn luyện.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR))
print(f"Project: {PROJECT_DIR}")

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torchvision import transforms
from torchvision.models import MobileNet_V2_Weights, mobilenet_v2
from tqdm.auto import tqdm

RESULTS_DIR = PROJECT_DIR / "outputs/results"
MODELS_DIR = PROJECT_DIR / "outputs/models"
PROCESSED_CSV_PATH = RESULTS_DIR / "processed_data.csv"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class ProcessedImageDataset(Dataset):
    def __init__(self, dataframe, project_dir, transform=None):
        self.paths = dataframe["processed_path"].astype(str).tolist()
        self.labels = dataframe["class_id"].astype(int).tolist()
        self.project_dir = Path(project_dir)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        relative_path = self.paths[index].replace("\\", "/")
        image_path = self.project_dir.joinpath(*relative_path.split("/"))
        with Image.open(image_path) as image:
            image = image.convert("RGB")
            if self.transform is not None:
                image = self.transform(image)
            else:
                image = image.copy()
        return image, self.labels[index]


def build_model(num_classes, weights=MobileNet_V2_Weights.DEFAULT):
    if num_classes < 2:
        raise ValueError("num_classes must be at least 2")
    model = mobilenet_v2(weights=weights)
    model.classifier[1] = nn.Linear(
        model.classifier[1].in_features, num_classes
    )
    return model


def run_epoch(model, loader, criterion, device, optimizer=None, description=None):
    training = optimizer is not None
    model.train(training)
    if training:
        model.features.eval()

    total_loss = total_correct = total_items = 0
    batches = tqdm(loader, desc=description, leave=False) if description else loader
    for images, labels in batches:
        images, labels = images.to(device), labels.to(device)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_items += labels.size(0)
        if description:
            batches.set_postfix(loss=f"{total_loss / total_items:.4f}")

    if total_items == 0:
        raise ValueError("The data loader is empty")
    return total_loss / total_items, total_correct / total_items


def train_model(model, train_loader, validation_loader, criterion, device,
                epochs, learning_rate, phase, checkpoint_path, config):
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=learning_rate,
        weight_decay=1e-4,
    )
    best_key = (-1.0, float("inf"))
    best_metrics = None
    history = []

    for epoch in range(1, epochs + 1):
        train_loss, train_accuracy = run_epoch(
            model, train_loader, criterion, device, optimizer,
            f"{phase} train {epoch}/{epochs}",
        )
        validation_loss, validation_accuracy = run_epoch(
            model, validation_loader, criterion, device,
            description=f"{phase} val {epoch}/{epochs}",
        )
        record = {
            "phase": phase,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "validation_loss": validation_loss,
            "validation_accuracy": validation_accuracy,
        }
        history.append(record)
        key = (validation_accuracy, -validation_loss)
        if key > best_key:
            best_key = key
            best_metrics = record
            checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save({
                "model_state": model.state_dict(),
                "class_names": config["class_names"],
                "phase": phase,
                "epoch": epoch,
                "metrics": record,
                "config": config,
            }, checkpoint_path)
        print(
            f"{phase} epoch {epoch}/{epochs} | "
            f"train loss {train_loss:.4f} | "
            f"val loss {validation_loss:.4f} | "
            f"val acc {validation_accuracy:.4f}"
        )
    return history, best_metrics


@torch.inference_mode()
def evaluate_model(model, loader, class_names, device, output_dir, prefix):
    model.eval()
    labels, predictions = [], []
    for images, batch_labels in loader:
        predictions.extend(model(images.to(device)).argmax(1).cpu().tolist())
        labels.extend(batch_labels.tolist())

    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    report = classification_report(
        labels, predictions, target_names=class_names,
        output_dict=True, zero_division=0,
    )
    matrix = confusion_matrix(labels, predictions, labels=range(len(class_names)))
    metrics = {
        "accuracy": float((labels == predictions).mean()),
        "macro_f1": float(f1_score(
            labels, predictions, labels=range(len(class_names)),
            average="macro", zero_division=0,
        )),
        "classification_report": report,
        "confusion_matrix": matrix.tolist(),
    }
    output_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(report).transpose().to_json(
        output_dir / f"{prefix}_classification_report.json", indent=2
    )
    pd.DataFrame(matrix).to_csv(
        output_dir / f"{prefix}_confusion_matrix.csv", index=False
    )
    return metrics


## Cấu hình

In [ ]:
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PHASE1_EPOCHS, PHASE2_EPOCHS = 5, 5
PHASE1_LR, PHASE2_LR = 1e-3, 1e-4
UNFREEZE_LAST_BLOCKS = 3
BATCH_SIZE, NUM_WORKERS = 32, 0
set_seed(SEED)
print(f"Device: {DEVICE}")

## Nạp dữ liệu đã resize và tạo DataLoader

Notebook đọc trực tiếp `outputs/results/processed_data.csv` bằng một `Dataset` nhỏ. Augmentation chỉ áp dụng cho train, còn normalization dùng đúng mean/std của MobileNetV2 pretrained.

In [ ]:
if not PROCESSED_CSV_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {PROCESSED_CSV_PATH}. Hãy chạy 02_preprocessing.ipynb trước."
    )

class_names = json.loads(
    (RESULTS_DIR / "class_names.json").read_text(encoding="utf-8")
)
processed_df = pd.read_csv(PROCESSED_CSV_PATH)
required_columns = {"processed_path", "class_name", "class_id", "split"}
missing_columns = required_columns - set(processed_df.columns)
if missing_columns:
    raise ValueError(f"processed_data.csv thiếu cột: {sorted(missing_columns)}")
class_to_idx = {name: index for index, name in enumerate(class_names)}
expected_ids = processed_df["class_name"].map(class_to_idx)
if expected_ids.isna().any() or not np.array_equal(
    processed_df["class_id"].to_numpy(), expected_ids.to_numpy()
):
    raise ValueError("class_id không khớp class_names.json.")

weights_preprocessing = MobileNet_V2_Weights.DEFAULT.transforms()
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=weights_preprocessing.mean, std=weights_preprocessing.std
    ),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=weights_preprocessing.mean, std=weights_preprocessing.std
    ),
])

def make_dataset(split_name, transform):
    subset = processed_df[processed_df["split"] == split_name].reset_index(drop=True)
    if subset.empty:
        raise ValueError(f"Split {split_name} không có dữ liệu.")
    return ProcessedImageDataset(subset, PROJECT_DIR, transform)


train_dataset = make_dataset("train", train_transform)
validation_dataset = make_dataset("validation", eval_transform)
test_dataset = make_dataset("test", eval_transform)
datasets_by_split = {
    "train": train_dataset,
    "validation": validation_dataset,
    "test": test_dataset,
}
split_counts = {name: len(dataset) for name, dataset in datasets_by_split.items()}
loader_options = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": DEVICE.type == "cuda",
}
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset, shuffle=True, generator=loader_generator, **loader_options
)
validation_loader = DataLoader(
    validation_dataset, shuffle=False, **loader_options
)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)

print(f"Classes: {len(class_names)}")
print("Split counts:", split_counts)

## Xây dựng model và cấu hình checkpoint

In [ ]:
weights = MobileNet_V2_Weights.DEFAULT
model = build_model(len(class_names), weights=weights).to(DEVICE)
criterion = torch.nn.CrossEntropyLoss()
config = {
    "seed": SEED,
    "device": str(DEVICE),
    "data_manifest": str(PROCESSED_CSV_PATH),
    "class_names": class_names,
    "num_classes": len(class_names),
    "image_size": [224, 224],
    "pretrained_weights": "MobileNet_V2_Weights.DEFAULT",
    "split_counts": split_counts,
    "phase_1": {"epochs": PHASE1_EPOCHS, "learning_rate": PHASE1_LR},
    "phase_2": {
        "epochs": PHASE2_EPOCHS,
        "learning_rate": PHASE2_LR,
        "unfreeze_last_blocks": UNFREEZE_LAST_BLOCKS,
    },
}
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(model.classifier)


## Phase 1 - chỉ huấn luyện classifier

In [ ]:
for parameter in model.features.parameters():
    parameter.requires_grad = False
for parameter in model.classifier.parameters():
    parameter.requires_grad = True

phase1_history, phase1_best = train_model(
    model, train_loader, validation_loader, criterion, DEVICE,
    PHASE1_EPOCHS, PHASE1_LR, "transfer_learning",
    MODELS_DIR / "mobilenet_v2_phase1_best.pt", config,
)
print("Kết quả tốt nhất phase 1:", phase1_best)


## Phase 2 - fine-tune các block cuối

In [ ]:
checkpoint = torch.load(
    MODELS_DIR / "mobilenet_v2_phase1_best.pt",
    map_location=DEVICE, weights_only=False,
)
model.load_state_dict(checkpoint["model_state"])

for parameter in model.features.parameters():
    parameter.requires_grad = False
for block in model.features[-UNFREEZE_LAST_BLOCKS:]:
    for parameter in block.parameters():
        parameter.requires_grad = True
for parameter in model.classifier.parameters():
    parameter.requires_grad = True

phase2_history, phase2_best = train_model(
    model, train_loader, validation_loader, criterion, DEVICE,
    PHASE2_EPOCHS, PHASE2_LR, "fine_tuning",
    MODELS_DIR / "mobilenet_v2_phase2_best.pt", config,
)
print("Kết quả tốt nhất phase 2:", phase2_best)


## Chọn checkpoint và đánh giá trên tập test

In [ ]:
use_phase2 = phase2_best["validation_accuracy"] >= phase1_best["validation_accuracy"]
chosen = phase2_best if use_phase2 else phase1_best
checkpoint_name = "mobilenet_v2_phase2_best.pt" if use_phase2 else "mobilenet_v2_phase1_best.pt"
chosen_path = MODELS_DIR / checkpoint_name
checkpoint = torch.load(chosen_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state"])

metrics = evaluate_model(
    model, test_loader, class_names, DEVICE,
    RESULTS_DIR, "mobilenet_v2_test",
)
history = pd.DataFrame(phase1_history + phase2_history)
history.to_csv(RESULTS_DIR / "mobilenet_v2_history.csv", index=False)

config["selected_checkpoint"] = str(chosen_path)
config["selected_validation_metrics"] = chosen
config["test_metrics"] = {
    "accuracy": metrics["accuracy"],
    "macro_f1": metrics["macro_f1"],
}
(RESULTS_DIR / "mobilenet_v2_config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Checkpoint: {chosen_path}")
print(f"Test accuracy: {metrics['accuracy']:.4f}")
print(f"Test macro F1: {metrics['macro_f1']:.4f}")


## Đường cong huấn luyện

In [ ]:
import matplotlib.pyplot as plt
for phase, values in history.groupby("phase", sort=False):
    plt.plot(values["epoch"], values["validation_accuracy"], marker="o", label=f"{phase} validation")
plt.title("Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## Báo cáo phân loại và ma trận nhầm lẫn

In [ ]:
report = pd.DataFrame(metrics["classification_report"]).transpose()
display(report)

plt.figure(figsize=(12, 10))
plt.imshow(metrics["confusion_matrix"], cmap="Blues")
plt.colorbar(label="So mau")
plt.title("MobileNetV2 - Ma tran nham lan tren test")
plt.xlabel("Nhan du doan")
plt.ylabel("Nhan that")
plt.xticks(range(len(class_names)), class_names, rotation=90, fontsize=6)
plt.yticks(range(len(class_names)), class_names, fontsize=6)
plt.tight_layout()
plt.show()